# Day 3 – Practical Lab: Speech-to-Text Transcription using OpenAI API

**Module:** Day 3 – Module 03: Introduction to Large Language Models
**Topic:** Task-Based Text AI LLMs – Automatic Speech Recognition (ASR)

> **Simulated business scenario:** The recording used in this lab is a training sample. A computer-generated voice reads a made-up IT incident update. It contains no real incident, customer or personal data.

## Section 1 – Lab Introduction

### What is Speech-to-Text?
**Speech-to-Text (STT)** converts spoken words in an audio recording into written text. Normally a person listens to a call or meeting and types what was said. Speech-to-Text does this job automatically.

### What is Automatic Speech Recognition (ASR)?
**Automatic Speech Recognition (ASR)** is the technology behind Speech-to-Text. It is the way a computer recognises the words in human speech without a person typing them.

### How does an audio transcription model work?
In simple terms, the model:
1. Receives the recording and breaks it into very short slices of sound.
2. Recognises the most likely words in each slice.
3. Uses the surrounding words to choose between similar-sounding options (for example "there" or "their").
4. Arranges the words into sentences and adds punctuation.

### Business applications of Speech-to-Text
- IT service desk call transcription
- Incident discussion transcription
- Meeting transcription
- Operations handover recordings
- Customer support call transcription

### Speech-to-Text vs Text-to-Speech

| | Speech-to-Text (STT) | Text-to-Speech (TTS) |
|---|---|---|
| Input | Audio recording | Written text |
| Output | Written text | Spoken audio |
| Example | A call recording becomes a transcript | An email is read aloud |

**This lab uses Speech-to-Text only.**

### Learning objectives
By the end of this lab you will be able to:
1. Explain what Speech-to-Text and Automatic Speech Recognition mean.
2. Load an audio file and check that it is valid.
3. Play a recording inside the notebook.
4. Connect to the OpenAI API using an API key.
5. Transcribe spoken English into text with the `gpt-transcribe` model.
6. Provide contextual hints for technical terms.
7. Save transcripts to files.
8. Process another audio recording.

### Basic workflow

```text
Audio Recording
      ↓
Python Audio File Handling
      ↓
OpenAI Transcription API
      ↓
Speech Recognition Model
      ↓
Generated Text Transcript
      ↓
Save Transcript
```

### Expected output
- The sample recording plays inside the notebook.
- A text transcript of the recording appears after the transcription cells run.
- Transcript files are saved in the `transcripts` folder, for example `incident_transcript.txt` and `incident_transcript_with_context.txt`.

**How to use this notebook:** run the cells from top to bottom. Click a cell and press **Shift + Enter** to run it. Do not skip cells.

## Section 2 – Environment Setup

This lab uses four packages. They are installed once from the file `day3_asr_requirements.txt` (see the Quickstart guide).

| Package | What it does in this lab |
|---|---|
| `openai` | Sends the audio to OpenAI and receives the text back |
| `jupyter` | Provides the notebook environment you are using now |
| `ipykernel` | Lets Jupyter run Python code in the notebook cells |
| `ipython` | Plays the audio and displays results inside the notebook |

The next code cell imports the tools this lab needs, creates the `transcripts` folder if it does not exist yet and confirms that the environment is ready. The other imports are standard Python tools that need no installation.

**Expected output:** the message `Environment ready.`, the OpenAI library version and the full path of the `transcripts` folder.

In [1]:
# Standard Python tools
import os                                    # reads environment variables such as OPENAI_API_KEY
from getpass import getpass                  # asks for the API key with a hidden input box
from pathlib import Path                     # works with file names and folders
from datetime import datetime                # creates unique file names with date and time

# Libraries installed from day3_asr_requirements.txt
import openai                                # OpenAI library (version and error types)
from openai import OpenAI                    # the client that talks to the OpenAI API
from IPython.display import Audio, Markdown, display   # play audio and show formatted output

# Folder for all transcripts (created automatically if it does not exist)
TRANSCRIPT_FOLDER = Path("transcripts")
TRANSCRIPT_FOLDER.mkdir(exist_ok=True)

print("Environment ready.")
print("OpenAI library version:", openai.__version__)
print("Transcripts folder:", TRANSCRIPT_FOLDER.resolve())

Environment ready.
OpenAI library version: 2.52.0
Transcripts folder: C:\D\Koenig\gen_ai_courses\Kyndryl\GenAI\labs\Day 4-rev\Day 4\Day 4 - Module 4 - Lab 2\Day 4 - Module 4 - Lab 2\transcripts


## Section 3 – OpenAI API Configuration

**What is an API?** An API (Application Programming Interface) is a set of rules that lets one program ask another program for a service. In this lab, Python sends the audio file to an OpenAI transcription model and receives the text back.

**Why is an API key needed?** The API key identifies your account. OpenAI uses it to check access and to record usage. You create the key on the **OpenAI API platform**. It is separate from a ChatGPT subscription.

**How does Python communicate with the model?**
1. This notebook creates an `OpenAI` client, which is a connection helper.
2. The client sends the audio file over an encrypted internet connection.
3. The transcription model returns the text in its response.

**Why must API keys be protected?** Anyone who has your key can use your account, and the usage is charged to you. Never paste the key into a notebook cell, a shared document or an email, and never publish it.

The next code cell reads the key from the environment variable `OPENAI_API_KEY`. If the variable is not set, it asks for the key in a hidden input box. The key is never printed on screen. The key is not written into the notebook.

**Expected output:** `OpenAI client is ready.` If no key is available, a configuration message explains what to do.

In [2]:
# 1. Read the API key from the OPENAI_API_KEY environment variable
api_key = os.environ.get("OPENAI_API_KEY")

# 2. If the variable is not set, ask for the key in a hidden input box
if not api_key:
    print("OPENAI_API_KEY is not set in this environment.")
    api_key = getpass("Paste your OpenAI API key and press Enter (input is hidden): ")

# 3. Create the client only when a key is available
if api_key:
    client = OpenAI(api_key=api_key)
    print("OpenAI client is ready. The API key is hidden for safety.")
else:
    client = None
    print("CONFIGURATION REQUIRED: no API key was provided.")
    print("Set the OPENAI_API_KEY environment variable (see the Quickstart guide), then run this cell again.")

# 4. Remove the key from this variable; the client keeps what it needs
api_key = None

# Shared variables used by later cells (they are filled in when a transcription succeeds)
transcript_text = None
transcript_with_context = None


def explain_error(error):
    """Turn an API error into a short message that participants can understand."""
    if isinstance(error, openai.AuthenticationError):
        return "The API key was rejected. Check that OPENAI_API_KEY holds a valid key."
    if isinstance(error, openai.APIConnectionError):
        return "Could not reach OpenAI. Check your internet connection and try again."
    if isinstance(error, openai.APIStatusError):
        return f"OpenAI returned an error (status {error.status_code}): {error.message}"
    return f"Unexpected problem: {error}"

OpenAI client is ready. The API key is hidden for safety.


## Sections 4 and 5 – Understanding and Listening to the Audio File

**Sample recording: "IT Infrastructure Incident Discussion"**
This is a simulated business scenario for training. The recording is an incident update from an IT infrastructure operations team.

| Property | Value |
|---|---|
| File | `sample_incident_audio.wav` |
| Format | WAV (uncompressed audio) |
| Length | about 49 seconds |
| Size | about 2 MB (well below the 25 MB limit) |
| Speech | One computer-generated English voice at a natural speaking speed |

**Expected transcription (summary):** the update reports that at about 10:15 AM users started to see intermittent connectivity problems with the corporate application. The application server CPU utilization reached 95 percent, and database connection latency increased. The service was restarted but the issue is still under investigation. About 120 users are affected. The incident is high priority and has been escalated. The next update follows within 30 minutes.

**Why listen first?** Before you judge a transcript, you need to know what was actually said. Listening first helps you spot errors quickly in Sections 8 and 12.

**Supported formats (OpenAI transcription API):** WAV, MP3, M4A, MP4, MPEG, MPGA, FLAC, OGG and WebM. The maximum file size is **25 MB**.

The next code cell checks that the file exists, that its format is supported and that its size is within the limit. If all checks pass, it shows the file details and an audio player.

**Expected output:** the file name, format and size, the message `File check passed.` and a play button. Press play to listen to the recording.

In [3]:
# The audio file to transcribe (keep it in the same folder as this notebook)
AUDIO_FILE_PATH = "sample_incident_audio.wav"

# Formats accepted by the OpenAI transcription API
SUPPORTED_FORMATS = [".flac", ".mp3", ".mp4", ".mpeg", ".mpga", ".m4a", ".ogg", ".wav", ".webm"]
MAX_FILE_SIZE_MB = 25


def check_audio_file(file_path):
    """Check that the file exists, has a supported format and is not too large."""
    path = Path(file_path)
    if not path.is_file():
        print(f"FILE NOT FOUND: '{file_path}'. Check the file name and keep the file in the notebook folder.")
        return False
    if path.suffix.lower() not in SUPPORTED_FORMATS:
        print(f"UNSUPPORTED FORMAT: '{path.suffix}'. Use one of: {', '.join(SUPPORTED_FORMATS)}")
        return False
    size_mb = path.stat().st_size / (1024 * 1024)
    if size_mb > MAX_FILE_SIZE_MB:
        print(f"FILE TOO LARGE: {size_mb:.1f} MB. The limit is {MAX_FILE_SIZE_MB} MB.")
        return False
    print(f"File name : {path.name}")
    print(f"Format    : {path.suffix.lower()}")
    print(f"File size : {size_mb:.2f} MB (limit {MAX_FILE_SIZE_MB} MB)")
    print("File check passed.")
    return True


# Check the file, then show it in the notebook with a play button
if check_audio_file(AUDIO_FILE_PATH):
    display(Audio(AUDIO_FILE_PATH))

File name : sample_incident_audio.wav
Format    : .wav
File size : 2.05 MB (limit 25 MB)
File check passed.


## Sections 6 and 7 – The OpenAI Transcription API and the First Transcription

**How the transcription process works**
1. **Audio upload:** the audio file is sent to OpenAI over an encrypted connection.
2. **Model selection:** you choose the model that listens to the audio.
3. **Transcription request:** Python calls `client.audio.transcriptions.create()` with the model and the file.
4. **API response:** OpenAI returns a response object.
5. **Extracting the text:** the written words are in `transcription.text`.

**Recommended model: `gpt-transcribe`**
`gpt-transcribe` is the model OpenAI currently recommends for transcribing completed recordings. This lab does not use `whisper-1` as the default because it is deprecated. Other older speech models are also marked deprecated.

**Official documentation** (check these for the latest details):
- Speech-to-text guide: https://developers.openai.com/api/docs/guides/speech-to-text
- Python reference for transcription: https://developers.openai.com/api/reference/python/resources/audio/subresources/transcriptions/methods/create

**What the next code cell does:** it sends the sample recording with the simplest possible request (model and file only) and displays the transcript. The `with open` pattern makes sure the audio file is closed again after the upload.

**Expected output:** a `--- TRANSCRIPT ---` heading followed by the spoken text. It should be close to the script described in Section 5. If the call fails, a plain-language message explains why. No sample text is shown in its place.

In [4]:
# Run only when an API key is available and the audio file is valid
if client is None:
    print("No API key available. Complete Section 3 first.")
elif check_audio_file(AUDIO_FILE_PATH):
    print("Sending the recording to OpenAI. This can take up to a minute...")
    try:
        # 'with open' opens the file and closes it automatically afterwards
        with open(AUDIO_FILE_PATH, "rb") as audio_file:
            transcription = client.audio.transcriptions.create(
                model="gpt-transcribe",   # the recommended transcription model
                file=audio_file,          # the audio recording to transcribe
            )
        transcript_text = transcription.text
        print("\n--- TRANSCRIPT ---")
        print(transcript_text)
    except Exception as error:
        transcript_text = None
        print("Transcription failed:", explain_error(error))

File name : sample_incident_audio.wav
Format    : .wav
File size : 2.05 MB (limit 25 MB)
File check passed.
Sending the recording to OpenAI. This can take up to a minute...

--- TRANSCRIPT ---
Good morning. This is an incident update from the IT Infrastructure Operations team. At approximately 10:15 a.m., users started reporting intermittent connectivity issues with the corporate application. Initial monitoring indicates that the application server CPU utilization has increased to 95 percent. The database server is also reporting increased connection latency. The infrastructure team has restarted the application service, but the issue is still under investigation. Approximately 120 users are currently impacted. The incident has been classified as high priority and escalated to the application support and database administration teams. The next status update will be shared within 30 minutes.


## Sections 8 and 9 – Understanding the Output and Saving the Transcript

**What to check in a transcript**
- **Recognised words:** do the words match what you heard? Look for words replaced by similar-sounding words.
- **Sentence structure:** are the sentences complete and in the right order?
- **Punctuation:** the model adds full stops and commas. Punctuation is not spoken, so these marks are the model's best estimate.
- **Technical terminology:** check terms such as "CPU utilization", "database", "latency" and "application server".
- **Numbers:** check values such as 10:15 AM, 95 percent, 120 users and 30 minutes. Numbers are a common source of errors.
- **Possible errors:** words may be missed, added or misheard, especially with background noise, fast speech or unusual names.

The character and word counts give a quick check of length. The sample script contains 99 words, so the word count should be close to 99.

**Why save transcripts?** A saved transcript can be searched, shared, reviewed or used in the next step of a business process. You also do not need to call the API again to read it.

The next two code cells show the counts and the readable transcript. Then they save the transcript to `transcripts/incident_transcript.txt` and read the file back to prove that it was saved.

**Note:** running the save cell again replaces `incident_transcript.txt` with the new transcript. Your own recordings are saved with unique names and are never overwritten (Section 13).

**Expected output:** the character count, the word count, the transcript shown as a quote, the full saved file location and the same text read back from the file.

In [5]:
if transcript_text is None:
    print("No transcript yet. Run the transcription cell in Section 7 first.")
else:
    print("Character count:", len(transcript_text))
    print("Word count     :", len(transcript_text.split()))   # words are separated by spaces
    print("\nTranscript:")
    display(Markdown("> " + transcript_text))                   # shown as a readable quote

Character count: 710
Word count     : 99

Transcript:


> Good morning. This is an incident update from the IT Infrastructure Operations team. At approximately 10:15 a.m., users started reporting intermittent connectivity issues with the corporate application. Initial monitoring indicates that the application server CPU utilization has increased to 95 percent. The database server is also reporting increased connection latency. The infrastructure team has restarted the application service, but the issue is still under investigation. Approximately 120 users are currently impacted. The incident has been classified as high priority and escalated to the application support and database administration teams. The next status update will be shared within 30 minutes.

In [6]:
def save_transcript(file_name, text):
    """Save text into the transcripts folder and return the full file path."""
    file_path = TRANSCRIPT_FOLDER / file_name
    with open(file_path, "w", encoding="utf-8") as output_file:
        output_file.write(text)
    return file_path


if transcript_text is None:
    print("Nothing to save yet. Run the transcription cell in Section 7 first.")
else:
    saved_path = save_transcript("incident_transcript.txt", transcript_text)
    print("Saved to:", saved_path.resolve())

    # Read the file back to prove that the transcript was saved
    with open(saved_path, "r", encoding="utf-8") as saved_file:
        print("\n--- CONTENT OF THE SAVED FILE ---")
        print(saved_file.read())

Saved to: C:\D\Koenig\gen_ai_courses\Kyndryl\GenAI\labs\Day 4-rev\Day 4\Day 4 - Module 4 - Lab 2\Day 4 - Module 4 - Lab 2\transcripts\incident_transcript.txt

--- CONTENT OF THE SAVED FILE ---
Good morning. This is an incident update from the IT Infrastructure Operations team. At approximately 10:15 a.m., users started reporting intermittent connectivity issues with the corporate application. Initial monitoring indicates that the application server CPU utilization has increased to 95 percent. The database server is also reporting increased connection latency. The infrastructure team has restarted the application service, but the issue is still under investigation. Approximately 120 users are currently impacted. The incident has been classified as high priority and escalated to the application support and database administration teams. The next status update will be shared within 30 minutes.


## Section 10 – Transcription Context and Technical Keywords

Speech models are trained on everyday language. In an IT environment they may hear specialised terms such as:

- Kubernetes
- PostgreSQL
- Firewall
- VPN
- DNS
- CPU utilization
- Database failover
- Network latency
- Load balancer
- Infrastructure monitoring

Without help, a model may write these terms differently, for example "V P N" or "vee pee en", or it may misspell an uncommon name. Contextual hints give the model advance information about the subject of the recording.

**The three settings used in this lab:**

| Setting | Sent to the API as | What it is for | Example |
|---|---|---|---|
| `TRANSCRIPTION_CONTEXT` | `prompt` | A short description of the recording and its terminology | "This recording relates to enterprise IT infrastructure operations…" |
| `TECHNICAL_KEYWORDS` | `keywords` | A list of exact terms you expect to hear | `"CPU utilization"`, `"VPN"` |
| `LANGUAGE_CODE` | `languages` | The language that is spoken | `"en"` (English) |

**Important:** the transcription context is guidance for speech recognition. It is **not** a normal ChatGPT conversation. Write it as a description of the recording, not as a question or a request for an answer.

**Important for this sample:** the sample recording does **not** mention PostgreSQL, Kubernetes, VPN, DNS or firewall. They are included because they are typical IT vocabulary. If these words appear in the context-assisted transcript, the hints have made the model write words that were never spoken. Check for this in Section 12.

The next code cell defines these settings. You can change them later to test how different hints affect the result.

**Expected output:** a confirmation message with the number of keywords and the language option that will be used.

In [7]:
# ---- Transcription context: a description of the recording (sent as "prompt") ----
TRANSCRIPTION_CONTEXT = """
This recording relates to enterprise IT infrastructure operations.
It may contain technical terminology including PostgreSQL,
Kubernetes, VPN, DNS, firewall, CPU utilization,
database failover and network latency.
Preserve technical terms accurately.
"""

# ---- Technical keywords: exact terms to expect (sent as "keywords") ----
TECHNICAL_KEYWORDS = [
    "PostgreSQL",
    "Kubernetes",
    "VPN",
    "DNS",
    "firewall",
    "CPU utilization",
    "database failover",
    "network latency"
]

# ---- Language: "en" = English. Leave empty ("") to let the model detect the language ----
LANGUAGE_CODE = "en"

# Language option used in the API calls (an empty dictionary means automatic detection)
language_options = {"languages": [LANGUAGE_CODE]} if LANGUAGE_CODE else {}

print("Context, keyword and language settings are ready.")
print("Number of keywords:", len(TECHNICAL_KEYWORDS))
print("Language option:", language_options if language_options else "automatic detection")

Context, keyword and language settings are ready.
Number of keywords: 8
Language option: {'languages': ['en']}


## Sections 13 and 14 – Processing Another Audio File and Language Settings

### Using your own recording
1. Put your audio file in the same folder as this notebook, for example `my_meeting.mp3`.
2. In the next code cell, change `CUSTOM_AUDIO_FILE` to your file name. Keep the quotation marks.
3. Run the cell. The file is checked, transcribed, shown on screen and saved to the `transcripts` folder.

Your transcript is saved with a unique name that combines the file name with the date and time, for example `custom_my_meeting_20261117_103000.txt`. Earlier transcripts are never overwritten.

If your file is larger than 25 MB, compress it (for example to MP3) or split it into shorter files.

**Note:** until you change the file name, the cell reports `FILE NOT FOUND` for the placeholder `your_audio_file.wav`. This is expected.

### Language settings
The model supports many languages. There are two ways to handle language:

- **Automatic language detection:** leave `LANGUAGE_CODE` empty (`""`). The model works out the language itself. This is useful when the language is unknown or mixed.
- **Language hint:** set `LANGUAGE_CODE` to a language code such as `"en"` (English), `"de"` (German) or `"fr"` (French). A correct hint can improve accuracy and speed. A wrong hint, for example `"de"` for English speech, gives poor results.

This lab transcribes English only and does not translate. The transcript is written in the language that was spoken. `LANGUAGE_CODE` was defined in Section 10 so that every transcription call uses it. Change it there to test automatic detection.

**Expected output:** the transcript of your file and the saved file location, or a clear message if the file cannot be found or is not supported.

In [10]:
# CHANGE THIS to the name of your own recording (keep the file in the notebook folder)
CUSTOM_AUDIO_FILE = "your_audio_file.wav"

if client is None:
    print("No API key available. Complete Section 3 first.")
elif check_audio_file(CUSTOM_AUDIO_FILE):
    print("Sending your recording to OpenAI. Longer files take more time...")
    try:
        with open(CUSTOM_AUDIO_FILE, "rb") as audio_file:
            transcription = client.audio.transcriptions.create(
                model="gpt-transcribe",
                file=audio_file,
                **language_options,
            )
        custom_text = transcription.text
        print("\n--- YOUR TRANSCRIPT ---")
        print(custom_text)

        # Unique name: file name + date and time, so no earlier transcript is overwritten
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        custom_file_name = f"custom_{Path(CUSTOM_AUDIO_FILE).stem}_{timestamp}.txt"
        saved_path = save_transcript(custom_file_name, custom_text)
        print("\nSaved to:", saved_path.resolve())
    except Exception as error:
        print("Transcription failed:", explain_error(error))

FILE NOT FOUND: 'your_audio_file.wav'. Check the file name and keep the file in the notebook folder.


## Section 15 – Complete Transcription Workflow

This section brings all the steps together in one reusable function.

```text
Select Audio File
      ↓
Validate Audio File
      ↓
Configure Transcription Context
      ↓
Call OpenAI API
      ↓
Display Transcript
      ↓
Save Transcript
```

The function `transcribe_and_save(audio_file_path)` does the following:
1. Validates the file and checks that an API key is available.
2. Calls the OpenAI API with the context, keyword and language settings from Section 10.
3. Displays the transcript.
4. Saves the transcript to the `transcripts` folder with a unique name.
5. Returns the transcript text, so that later steps can use it.

You do not need to write or change this function. To process any file, call it with the file name, for example `transcribe_and_save("my_meeting.mp3")`.

If the file check or the API call fails, the function prints a message and returns `None`. It never invents a transcript.

**Expected output:** the complete workflow runs on the sample recording. The transcript is displayed, saved as a new file in `transcripts` and returned. This cell makes one more API call for the same recording, so it uses a small amount of API usage.

In [ ]:
def transcribe_and_save(audio_file_path):
    """Validate the file, transcribe it, show the text, save it and return the text."""
    # 1. Validate the file and check that an API key is available
    if client is None:
        print("No API key available. Complete Section 3 first.")
        return None
    if not check_audio_file(audio_file_path):
        return None

    # 2. Call the OpenAI API with the Section 10 settings
    try:
        with open(audio_file_path, "rb") as audio_file:
            transcription = client.audio.transcriptions.create(
                model="gpt-transcribe",
                file=audio_file,
                prompt=TRANSCRIPTION_CONTEXT,
                keywords=TECHNICAL_KEYWORDS,
                **language_options,
            )
    except Exception as error:
        print("Transcription failed:", explain_error(error))
        return None

    # 3. Display the transcript
    text = transcription.text
    print("--- TRANSCRIPT ---")
    print(text)

    # 4. Save the transcript with a unique name and return it
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    saved_path = save_transcript(f"{Path(audio_file_path).stem}_workflow_{timestamp}.txt", text)
    print("\nSaved to:", saved_path.resolve())
    return text


# Run the complete workflow on the sample recording
workflow_text = transcribe_and_save(AUDIO_FILE_PATH)

## Section 16 – Knowledge Check

1. What is Automatic Speech Recognition?
2. What is the purpose of the OpenAI transcription API?
3. What is the role of the `gpt-transcribe` model?
4. Why are contextual hints useful?
5. What is the difference between Speech-to-Text and Text-to-Speech?

### Answers
1. Automatic Speech Recognition (ASR) is the technology that lets a computer recognise the words in spoken audio and convert them into text.
2. The OpenAI transcription API lets a program send an audio file to OpenAI and receive a text transcript back, without a person typing it.
3. `gpt-transcribe` is the model that listens to the recording and produces the text. It is the model OpenAI currently recommends for completed audio files.
4. Hints about the subject, the expected terms and the language help the model recognise technical words and names. They do not guarantee better results, so the output must still be checked.
5. Speech-to-Text turns audio into text. Text-to-Speech turns text into audio.

## Section 17 – Lab Summary

In this lab you have:
- Loaded an audio file and checked it.
- Played the recording.
- Connected to the OpenAI API.
- Transcribed speech into text.
- Used contextual hints.
- Compared transcription results.
- Saved the generated transcripts.
- Processed another audio file.

### Practical enterprise applications
- **IT service desk call transcription:** a written record of support calls.
- **Incident discussion transcription:** a written record of incident calls for review.
- **Meeting transcription:** searchable notes from team meetings.
- **Operations handover recordings:** handover notes that the next shift can read.
- **Customer support call transcription:** written records of customer calls for quality checks.

### Optional troubleshooting

| Message you see | What to do |
|---|---|
| `CONFIGURATION REQUIRED: no API key` | Set `OPENAI_API_KEY` (see the Quickstart), restart Jupyter and run the notebook again from the top. |
| `FILE NOT FOUND` | Check the file name and make sure the file is in the same folder as the notebook. |
| `UNSUPPORTED FORMAT` | Convert the recording to MP3 or WAV. |
| `FILE TOO LARGE` | Compress the recording or split it into shorter files (limit 25 MB). |
| `The API key was rejected` | Check the key, or create a new key on the OpenAI API platform. |
| `Could not reach OpenAI` | Check your internet connection and run the cell again. |
| `OpenAI returned an error` | Read the message shown. Run the cell again later if the service is busy. |